In [1]:
import os
# FIX: Prevent the OpenMP framework from crashing the Jupyter Kernel on Mac
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import time
import pandas as pd
from pathlib import Path

# Add the project root to sys.path so we can safely load local data
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Modern Google GenAI Embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Modern Vector Store Imports
from langchain_community.vectorstores import FAISS
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# Set up the modern Google Embeddings model
GCP_PROJECT_ID = "gd-gcp-gridu-genai"

print("🧠 Initializing Google GenAI Embeddings...")
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004", 
    project=GCP_PROJECT_ID, 
    location="us-central1"
)

# Define local Database Paths
faiss_path = str(project_root / "data" / "vector_store" / "faiss_p1")
qdrant_path = str(project_root / "data" / "vector_store" / "qdrant_p1")

print("✅ Setup complete. Ready to load databases.")

/Users/pboodida/Desktop/GEN_AI-3_RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/pp/dqdqbjt97c59y9lw190p5yf00000gn/T/ipykernel_59726/3626991161.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


🧠 Initializing Google GenAI Embeddings...
✅ Setup complete. Ready to load databases.


In [2]:
print("⚡ Loading FAISS DB...")
# allow_dangerous_deserialization=True is required by LangChain for local FAISS files
faiss_db = FAISS.load_local(faiss_path, embeddings, allow_dangerous_deserialization=True)

print("⚡ Loading Qdrant DB...")
# With the new Qdrant integration, we first create the client, then attach it to the VectorStore
qdrant_client = QdrantClient(path=qdrant_path)
qdrant_db = QdrantVectorStore(
    client=qdrant_client, 
    collection_name="ifc_annual_report", 
    embedding=embeddings
)

# Test queries based on typical financial annual report questions
test_queries = [
    "What was the total net income for the fiscal year?",
    "Summarize the CEO's statement regarding emerging markets.",
    "What are the main risks associated with currency fluctuations?",
    "List the major investments in Latin America.",
    "What is the ESG framework and sustainability goals?"
]

print("✅ Databases loaded successfully. Ready for performance testing.")

⚡ Loading FAISS DB...
⚡ Loading Qdrant DB...
✅ Databases loaded successfully. Ready for performance testing.


In [3]:
results = []

print("⏳ Running Benchmarks. Please wait...\n")

for query in test_queries:
    # --- Benchmark FAISS ---
    faiss_start = time.perf_counter()
    faiss_docs = faiss_db.similarity_search(query, k=5)
    faiss_time = (time.perf_counter() - faiss_start) * 1000 # Convert to milliseconds
    
    # --- Benchmark Qdrant ---
    qdrant_start = time.perf_counter()
    qdrant_docs = qdrant_db.similarity_search(query, k=5)
    qdrant_time = (time.perf_counter() - qdrant_start) * 1000 # Convert to milliseconds
    
    # Store the results
    results.append({
        "Query": query[:40] + "...",
        "FAISS_Time_ms": round(faiss_time, 2),
        "Qdrant_Time_ms": round(qdrant_time, 2)
    })

# Display Results
df_results = pd.DataFrame(results)
display(df_results)

print("-" * 40)
print(f"📊 Average FAISS Retrieval:  {df_results['FAISS_Time_ms'].mean():.2f} ms")
print(f"📊 Average Qdrant Retrieval: {df_results['Qdrant_Time_ms'].mean():.2f} ms")

⏳ Running Benchmarks. Please wait...



,Query,FAISS_Time_ms,Qdrant_Time_ms
0,What was the total net income for the fi...,442.25,1215.38
1,Summarize the CEO's statement regarding ...,415.54,1235.64
2,What are the main risks associated with ...,424.08,1673.64
3,List the major investments in Latin Amer...,697.72,1052.47
4,What is the ESG framework and sustainabi...,568.45,1054.89


----------------------------------------
📊 Average FAISS Retrieval:  509.61 ms
📊 Average Qdrant Retrieval: 1246.40 ms
